# xarray / xarray-jax comparison

Three integration routes, two encodings of a ragged `PropertyMap`, and the
selector forms that survive translation.

In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr

ROOT = Path.cwd()
if not (ROOT / "src" / "summer4").is_dir():
    ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else ROOT
sys.path.insert(0, str(ROOT))

from explorations.datatypes.prototype import PropertyData
from explorations.datatypes.xr_bridge import (
    ABSENT_LABEL,
    blocks_to_datatree,
    from_blocks,
    from_dataarray,
    register_xarray_pytrees,
    selector_mask,
    to_blocks,
    to_dataarray,
)
from summer4 import Property, PropertyMap

print("jax", jax.__version__)
print("xarray", xr.__version__)
assert jax.__version__.startswith("0.6")
assert tuple(int(p) for p in xr.__version__.split(".")[:1])[0] >= 2026
assert hasattr(jnp.ones(1), "__array_namespace__")

## Route 1: PyPI `xarray-jax` 0.0.5

Recorded pins: `jax<0.5.0,>=0.4.33` and `xarray<2025.0.0`. Incompatible with
this environment. Confirmed against PyPI when the network is available.

In [ ]:
XARRAY_JAX_PINS = {
    "name": "xarray-jax",
    "version": "0.0.5",
    "requires": ["jax<0.5.0,>=0.4.33", "equinox<0.12.0,>=0.11.7", "xarray<2025.0.0,>=2024.9.0"],
}

try:
    importlib.metadata.version("xarray-jax")
    installed = True
except importlib.metadata.PackageNotFoundError:
    installed = False
assert installed is False, "xarray-jax must not be installed; it pins jax<0.5"

pypi_pins = None
try:
    with urllib.request.urlopen("https://pypi.org/pypi/xarray-jax/json", timeout=10) as resp:
        info = json.load(resp)["info"]
    pypi_pins = info["requires_dist"]
    print("pypi xarray-jax", info["version"], pypi_pins)
    assert any(req.startswith("jax<0.5.0") for req in pypi_pins)
except (urllib.error.URLError, TimeoutError, OSError) as exc:
    print("pypi lookup skipped:", exc)
    pypi_pins = XARRAY_JAX_PINS["requires"]

jax_pin = next(req for req in pypi_pins if req.startswith("jax"))
assert "<0.5" in jax_pin
print("PyPI xarray-jax is incompatible with jax", jax.__version__)

## Route 2: DeepMind `gdm-xarray-jax`

GitHub-only (`google-deepmind/xarray_jax`), not published to PyPI. Requires
`xarray>=2026.1.0`. We do not install it; the vendored registration below is
the same idea (pytree flatten of data leaves).

In [ ]:
GDM_RECORDED = {
    "name": "gdm-xarray-jax",
    "on_pypi": False,
    "requires_python": ">=3.12",
    "requires": ["jax>=0.4.30", "numpy>=1.24", "xarray>=2026.1.0"],
    "repo": "https://github.com/google-deepmind/xarray_jax",
}

gdm_on_pypi = None
try:
    urllib.request.urlopen("https://pypi.org/pypi/gdm-xarray-jax/json", timeout=10)
    gdm_on_pypi = True
except urllib.error.HTTPError as exc:
    gdm_on_pypi = exc.code != 404
    print("gdm-xarray-jax pypi status", exc.code)
except (urllib.error.URLError, TimeoutError, OSError) as exc:
    print("pypi lookup skipped:", exc)
    gdm_on_pypi = GDM_RECORDED["on_pypi"]

assert gdm_on_pypi is False
try:
    importlib.metadata.version("gdm-xarray-jax")
    gdm_installed = True
except importlib.metadata.PackageNotFoundError:
    gdm_installed = False
assert gdm_installed is False
print("gdm-xarray-jax is GitHub-only; not installed in this spike")

## Route 3: vendored pytree registration

JAX arrays already duck-type through xarray. Registering `DataArray` as a
pytree lets `jit` / `tree_map` see the buffer and restore labels.

In [ ]:
register_xarray_pytrees()

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
sev = Property("severity", ("mild", "severe"))
pm = PropertyMap.from_property(state).stratify(age).stratify(sev, where=state["I"])
values = jnp.arange(pm.size, dtype=jnp.float32)
pd = PropertyData.wrap(pm, values)

da = to_dataarray(pd)
assert da.dims == ("compartment",)
assert da.sizes["compartment"] == 12
assert "age" in da.coords and "severity" in da.coords

back = from_dataarray(da, pm)
assert jnp.allclose(back.data, pd.data)


@jax.jit
def double_da(arr):
    return arr * 2


out_da = double_da(da)
assert isinstance(out_da, xr.DataArray)
assert np.allclose(np.asarray(out_da.data), np.asarray(values) * 2)

mapped = jax.tree.map(lambda x: x + 1, da)
assert np.allclose(np.asarray(mapped.data), np.asarray(values) + 1)

## Selectors vs xarray indexers

Non-dimension coordinates do not give `.sel()` the Kleene algebra.
`severity == "mild"` is two-valued; `!= "mild"` includes absent rows, which
is the opposite of `~sev["mild"]`.

In [ ]:
age_coord = np.asarray(da.coords["age"].values)
sev_coord = np.asarray(da.coords["severity"].values)

xr_age = age_coord == "0-4"
pm_age = np.asarray(pm.mask(age["0-4"]))
assert np.array_equal(xr_age, pm_age), "simple equality matches a present trait"

xr_not_mild = sev_coord != "mild"
pm_not_mild = np.asarray(pm.mask(~sev["mild"]))
assert not np.array_equal(xr_not_mild, pm_not_mild), "xarray != includes Kleene-unknown"

xr_absent = sev_coord == ABSENT_LABEL
pm_absent = np.asarray(pm.mask(sev.absent()))
assert np.array_equal(xr_absent, pm_absent), "absent is a sentinel, not a first-class NA"

mask = selector_mask(da, pm, state["I"] & age["0-4"] & ~sev["severe"])
assert int(np.asarray(mask).sum()) == int(pm.select(state["I"] & age["0-4"] & ~sev["severe"]).size)
print("Kleene queries must go through PropertyMap.mask, not DataArray.sel")

## Presence-signature blocks

The ragged SIR+age+severity map splits into two dense cubes: `(state, age)`
for S/R and `(state, age, severity)` for I.

In [ ]:
blocks = to_blocks(pd)
assert set(blocks) == {("state", "age"), ("state", "age", "severity")}
sr = blocks[("state", "age")]
inf = blocks[("state", "age", "severity")]
assert sr.dims == ("state", "age")
assert inf.dims == ("state", "age", "severity")
assert sr.sizes["age"] == 3 and inf.sizes["severity"] == 2
# I is absent from the (state, age) block → NaN at state='I'.
assert np.isnan(sr.sel(state="I").values).all()
assert np.isfinite(sr.sel(state="S").values).all()

tree = blocks_to_datatree(blocks)
print("DataTree type", type(tree).__name__)

restored = from_blocks(blocks, pm)
assert jnp.allclose(restored.data, pd.data)

# Dense-block reduce along age vs PropertyData.segment_sum.
sr_sum = np.nansum(sr.values, axis=sr.get_axis_num("age"))
inf_sum = np.nansum(inf.values, axis=inf.get_axis_num("age"))
pd_sum = pd.sum_over(age)
assert pd_sum.data.shape == (3,)
print("block age-sums SR", sr_sum, "I", inf_sum, "flat", np.asarray(pd_sum.data))

## Round-trip and encoding cost

In [ ]:
def _ready(out):
    data = out.data if isinstance(out, (PropertyData, xr.DataArray)) else out
    if hasattr(data, "block_until_ready"):
        data.block_until_ready()
    return out


def bench(fn, *, n=20, warmup=3, name=""):
    for _ in range(warmup):
        _ready(fn())
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        _ready(fn())
        times.append(time.perf_counter() - t0)
    times.sort()
    med = times[len(times) // 2]
    print(f"{name:48s}  median={med * 1e6:9.1f} us")
    return med


@jax.jit
def pd_roundtrip(pd_in):
    return from_dataarray(to_dataarray(pd_in), pd_in.pmap)


@jax.jit
def pd_mul(pd_in):
    return pd_in * 2


@jax.jit
def da_mul(arr):
    return arr * 2


pd_roundtrip(pd)
pd_mul(pd)
da_mul(da)

RESULTS = {}
RESULTS["eager_to_da"] = bench(lambda: to_dataarray(pd), name="eager to_dataarray")
RESULTS["eager_from_da"] = bench(lambda: from_dataarray(da, pm), name="eager from_dataarray")
RESULTS["eager_to_blocks"] = bench(lambda: to_blocks(pd), name="eager to_blocks")
RESULTS["eager_from_blocks"] = bench(lambda: from_blocks(blocks, pm), name="eager from_blocks")
RESULTS["jit_pd_mul"] = bench(lambda: pd_mul(pd), name="jit PropertyData *2")
RESULTS["jit_da_mul"] = bench(lambda: da_mul(da), name="jit DataArray *2")
RESULTS["jit_roundtrip"] = bench(lambda: pd_roundtrip(pd), name="jit flat roundtrip")

rt = pd_roundtrip(pd)
assert jnp.allclose(rt.data, pd.data)
print("RESULTS_US", {k: round(v * 1e6, 2) for k, v in RESULTS.items()})

In [ ]:
assert RESULTS["jit_pd_mul"] > 0
assert RESULTS["jit_da_mul"] > 0
assert RESULTS["eager_to_blocks"] > 0
print(
    "vendored DataArray jit works; Kleene queries stay on PropertyMap; "
    "presence blocks are an export, not a replacement."
)